In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import numpy as np
import re

In [2]:
from data import UncertainDataset

In [3]:
from pathlib import Path

In [4]:
def compute_min_max(paramID: int, data: xr.Dataset):
    variable = None
    
    min_ = np.nanmin(data.values)  # ignore NANs in grid cells which are not classified
    max_ = np.nanmax(data.values)  
    
    return min_, max_

In [5]:
def collect_info(paramID: int, data: xr.Dataset, grib_info: dict, level):
    parameterName = data.attrs.get('long_name')
    shortName = data.attrs.get('GRIB_shortName')
    parameterUnits = "dimensionless" if re.fullmatch(r"\(?0\s*-\s*1\)?", data.attrs.get('GRIB_units')) else data.attrs.get('GRIB_units')
    bitsPerValue = grib_info[paramID]["bitsPerValue"]
    min_, max_ = compute_min_max(paramID, data)
    error = (max_ - min_) / 2 ** bitsPerValue
    print(f"{parameterName}, {shortName}, {paramID}, {parameterUnits}, {level}, {min_}, {max_}, {error}")
    return (parameterName, shortName, paramID, parameterUnits, level, min_, max_, error)
    
      

In [6]:
from tqdm import tqdm
import eccodes

In [7]:
import re

def atoi(s: str):
    return int(s) if s.isdigit() else s

def natural_keys(s: str) -> list[str | int]:
    '''
    alist.sort(key=natural_keys) sorts in human order
    http://nedbatchelder.com/blog/200712/human_sorting.html
    '''
    return [atoi(c) for c in re.split(r"\d+", s)]

In [8]:
def get_grib_information(level):
    data_path = Path("/ec/res4/scratch/ecm5945/era5-data")
    grib = data_path / f"{level}_data" / "data_20240101.grib"
    grib_info = {}
    with eccodes.highlevel.FileReader(grib) as reader:
        for message in reader:
            parameterName = re.sub(r" \($", "", message.get("parameterName"))
            shortName = message.get("shortName")
            paramID = message.get("paramId")
            if not paramID:
                paramID = message.get("ifsparam")
            parameterUnits = "dimensionless" if re.fullmatch(r"\(?0\s*-\s*1\)?", message.get("parameterUnits")) else message.get("parameterUnits")
            bitsPerValue = message.get("bitsPerValue")
            dataType = message.get("dataType")

    
            line = f""
            if dataType == "em":
                grib_info[paramID] = {"parameterName": parameterName, "shortName": shortName, "parameterUnits": parameterUnits, "bitsPerValue": bitsPerValue}
    
    return grib_info

In [9]:
from data import ds_single

grib_info_single = get_grib_information("single")

with open("single_info.txt", "w") as file:
    file.write("Field Name\tShort Name\tgrib paramID\tUnit\tLevel\tMinimum value\tMaximum value\tExisting L1 error\n")
    for key in ds_single.mean:
        paramID = ds_single.mean[key].attrs.get('GRIB_paramId')
        data = collect_info(paramID, ds_single.mean[key],grib_info_single, "Single/Surface")
        file.write("\t".join(map(str, data)) + "\n")

Surface runoff, sro, 8, m, Single/Surface, 0.0, 0.040673255920410156, 6.206246325746179e-07
Sub-surface runoff, ssro, 9, m, Single/Surface, 0.0, 0.02685689926147461, 4.098037607036531e-07
UV visible albedo for direct radiation (climatological), aluvp, 15, dimensionless, Single/Surface, 0.011121053248643875, 0.9175923466682434, 1.3831654541718308e-05
UV visible albedo for diffuse radiation (climatological), aluvd, 16, dimensionless, Single/Surface, 0.012219812721014023, 0.9175915122032166, 1.3814876183460001e-05
Near IR albedo for direct radiation (climatological), alnip, 17, dimensionless, Single/Surface, 0.05909455195069313, 0.6660874485969543, 9.261976629204582e-06
Near IR albedo for diffuse radiation (climatological), alnid, 18, dimensionless, Single/Surface, 0.059999994933605194, 0.6812921166419983, 9.48016531765461e-06
Lake cover, cl, 26, dimensionless, Single/Surface, 0.0, 1.0, 5.960464477539063e-08
Low vegetation cover, cvl, 27, dimensionless, Single/Surface, 0.0, 1.0, 5.9604644

In [ ]:
from data import ds_pressure

grib_info_pressure = get_grib_information("pressure")

with open("pressure_info.txt", "w") as file:
    file.write("Field Name\tShort Name\tgrib paramID\tUnit\tLevel\tMinimum value\tMaximum value\tExisting L1 error\n")
    for key in ds_pressure.mean:
        paramID = ds_pressure.mean[key].attrs.get('GRIB_paramId')
        data = collect_info(paramID, ds_pressure.mean[key], grib_info_pressure, "Pressure")
        file.write("\t".join(map(str, data)) + "\n")
